# AI Applications & Mini Project 1 — Predictive maintenance

**Digital Foundation for Chemical Engineers**  \nDuration: 60 minutes

## Today’s goal
Understand how sensor data can support maintenance decisions, explore a pump-data CSV, and build a transparent first-pass maintenance flag. Predictive maintenance supports—not replaces—inspection, maintenance expertise, and safety procedures.

## 1. From reactive to predictive maintenance (10 min)

- Reactive: repair after failure.
- Preventive: service on a fixed schedule.
- Predictive: use condition, operating, and history data to prioritise maintenance.

AI may find patterns across variables or predict future condition. Before using a model, we need trustworthy data, a defined target, and safe human decision-making. An algorithmic score is not a maintenance instruction.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

pump_data = pd.read_csv('https://raw.githubusercontent.com/aryapitnaik-a11y/Module_5_6_Both/refs/heads/main/data/quality_batches_120.csv')
pump_data['timestamp'] = pd.to_datetime(pump_data['timestamp'])
pump_data.head()

,timestamp,batch_id,reactor,inlet_temperature_c,conversion_percent,product_purity_percent,lab_result
0,2026-02-02 08:00:00,Q2001,R1,72.0,88.5,96.20,REVIEW
1,2026-02-02 09:00:00,Q2001,R2,72.6,88.5,96.39,REVIEW
2,2026-02-02 10:00:00,Q2001,R3,73.2,89.0,96.58,REVIEW
3,2026-02-02 11:00:00,Q2001,R1,73.8,89.9,96.77,REVIEW
4,2026-02-02 12:00:00,Q2001,R2,74.4,89.9,96.96,REVIEW


## 2. Inspect before making claims (10 min)

The CSV contains pump identity, vibration, bearing temperature, motor current, and running hours. Ask: What does each measurement mean? How often was it sampled? Is the sensor calibrated? Which inspection findings or failures would be needed to train a real model?

In [4]:
pump_data.info()
display(pump_data.describe())
print('Missing values:')
print(pump_data.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   timestamp               120 non-null    datetime64[ns]
 1   batch_id                120 non-null    object        
 2   reactor                 120 non-null    object        
 3   inlet_temperature_c     120 non-null    float64       
 4   conversion_percent      120 non-null    float64       
 5   product_purity_percent  120 non-null    float64       
 6   lab_result              120 non-null    object        
dtypes: datetime64[ns](1), float64(3), object(3)
memory usage: 6.7+ KB


,timestamp,inlet_temperature_c,conversion_percent,product_purity_percent
count,120,120.000000,120.000000,120.000000
mean,2026-02-04 19:30:00,74.355000,91.192500,97.043333
min,2026-02-02 08:00:00,72.000000,88.000000,96.200000
25%,2026-02-03 13:45:00,73.200000,89.475000,96.580000
50%,2026-02-04 19:30:00,74.400000,91.150000,96.960000
75%,2026-02-06 01:15:00,75.600000,92.800000,97.530000
max,2026-02-07 07:00:00,76.800000,94.700000,97.910000
std,NaN,1.563762,1.946821,0.549290


Missing values:
timestamp                 0
batch_id                  0
reactor                   0
inlet_temperature_c       0
conversion_percent        0
product_purity_percent    0
lab_result                0
dtype: int64


In [8]:
max_vibration=pump_data['vibration_mm_s'].max
print('maxmium vibration:'max_vibration)
pump_data[(pump_data['vibration_mm_s']


SyntaxError: invalid syntax. Perhaps you forgot a comma? (757066068.py, line 2)

**Try it (4 min):** Identify the maximum vibration and the pump on which it occurred.

**Challenge:** Group by pump_id and compare average vibration and bearing temperature.

## 3. Visual inspection of condition trends (12 min)

A trend chart is often the first useful maintenance tool. It can reveal drift, spikes, or changes after maintenance. A visual pattern alone does not establish a fault cause.

In [10]:
plt.figure(figsize=(9, 4))
for pump_id, group in pump_data.groupby('pump_id'):
    plt.plot(group['timestamp'], group['vibration_mm_s'], marker='.', label=pump_id)
plt.axhline(2.8, color='red', linestyle='--', label='Classroom review level')
plt.title('Pump vibration trend')
plt.xlabel('Time')
plt.ylabel('Vibration (mm/s)')
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

KeyError: 'pump_id'

<Figure size 900x400 with 0 Axes>

## 4. Transparent first-pass flag (15 min)

This is a rule-based screening tool, not AI. It helps show what a later ML model would need to improve on. The review levels are classroom examples only and must never be used as real operating limits.

In [11]:
vibration_review_mm_s = 2.8
bearing_review_c = 65
review_readings = pump_data[(pump_data['vibration_mm_s'] >= vibration_review_mm_s) |
                            (pump_data['bearing_temperature_c'] >= bearing_review_c)].copy()
display(review_readings)
print('Readings requiring engineering review:', len(review_readings))

KeyError: 'vibration_mm_s'

## Pair task (9 min)
Write a two-sentence maintenance note based on the flagged rows. Include the observation and evidence you would seek next.

**Challenge:** Add a review_reason column that distinguishes vibration-only, temperature-only, and both-condition flags.

## Exit check
You can explain the predictive-maintenance workflow and separate a transparent screening rule from an AI prediction. Next: AI-supported quality control.